# 001 Guardrails

这是 LangChain Advanced usage 学习线的第一份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/guardrails

学习目标：

1. 理解 guardrails 不是一个单独功能，而是一组运行时安全边界
2. 学习 built-in guardrails：PII 和 human-in-the-loop
3. 学习 before-agent 输入拦截
4. 学习 after-agent 输出拦截
5. 学习 tool-call 拦截
6. 学习多个 guardrails 如何组合
7. 对比本仓库 Harness 的 approval、allowed_paths、ledger、verification

这一讲全部使用 fake model 和确定性规则，不消耗真实模型额度，也不会执行危险工具。

## 1. Guardrails 是什么

Guardrails 可以先理解成 agent 的安全边界。

它不是让模型“更聪明”，而是限制 agent 在输入、模型调用、工具调用、输出这些位置不能越界。

用 Java 来类比：

```text
Guardrails
  ~= 请求校验 + 权限拦截器 + 输出过滤器 + 审计规则
```

常见 guardrail 类型：

| 类型 | 位置 | 例子 |
| --- | --- | --- |
| 输入 guardrail | agent 开始前 | 禁止危险请求、过滤敏感输入 |
| 模型 guardrail | 模型调用前后 | 上下文预算、模型输出检查 |
| 工具 guardrail | 工具执行前 | shell / delete / write_file 需要审批 |
| 输出 guardrail | agent 完成后 | 禁止泄露 secret、过滤不合规回答 |
| 人工审批 | 工具执行前或关键节点 | 高风险操作必须等用户确认 |

在 LangChain 里，guardrails 通常通过 middleware 实现。

In [11]:
from typing import Any

from langchain.agents import create_agent
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    HumanInTheLoopMiddleware,
    PIIMiddleware,
    after_agent,
    before_agent,
    hook_config,
    wrap_tool_call,
)
from langchain_core.language_models.fake_chat_models import FakeListChatModel, FakeMessagesListChatModel
from langchain_core.messages import AIMessage, RemoveMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.runtime import Runtime


## 2. Built-in Guardrail：PII 脱敏

`PIIMiddleware` 用来处理隐私信息，例如邮箱、银行卡、IP、MAC、URL 等。

它支持不同策略：

- `redact`：替换成占位符
- `mask`：部分打码
- `hash`：哈希化
- `block`：直接阻断

下面用 fake model 看输入里的邮箱是否会被脱敏。

In [15]:
pii_agent = create_agent(
    model=FakeListChatModel(responses=["我已经收到脱敏后的输入。"]),
    tools=[],
    middleware=[PIIMiddleware("email", strategy="mask", apply_to_input=True)],
)

pii_result = pii_agent.invoke(
    {"messages": [{"role": "user", "content": "我的邮箱是 java.user@example.com，请记一下。"}]}
)

for message in pii_result["messages"]:
    print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))


human 我的邮箱是 java.user@****.com，请记一下。
ai 我已经收到脱敏后的输入。


## 3. Built-in Guardrail：Human-in-the-loop

`HumanInTheLoopMiddleware` 用来在工具执行前中断，等待人工审批。

这和本仓库 Harness 的 `ApprovalTicket` 很像：

```text
模型提出 tool call
  -> middleware 判断风险
  -> 需要审批则中断
  -> 用户批准后恢复执行
```

这一节只创建 middleware，不真正执行审批流程。完整审批需要 checkpointer / resume 流程，适合后续单独一讲。

In [16]:
hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "delete_file": True,
        "shell": True,
        "write_file": True,
    },
    description_prefix="高风险工具执行需要人工审批",
)

print(type(hitl).__name__)
print(hitl)


HumanInTheLoopMiddleware


## 4. 输入 Guardrail：before-agent 拦截

输入 guardrail 适合在 agent 真正开始前阻断请求。

比如：

- 用户要求删除所有文件
- 用户要求泄露密钥
- 用户输入包含明确违规词

这里用 `before_agent(can_jump_to=["end"])` 做一个确定性拦截。

In [17]:
@before_agent(can_jump_to=["end"])
def block_dangerous_input(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    text = " ".join(str(getattr(message, "content", "")) for message in state.get("messages", []))
    lowered = text.lower()

    banned_keywords = ["delete all", "drop database", "泄露密钥"]
    if any(keyword in lowered for keyword in banned_keywords):
        return {
            "messages": [AIMessage(content="请求已被输入 guardrail 阻止：操作风险过高。")],
            "jump_to": "end",
        }

    return None


input_guardrail_agent = create_agent(
    model=FakeListChatModel(responses=["输入通过，模型正常回答。"]),
    tools=[],
    middleware=[block_dangerous_input],
)

for prompt in ["帮我总结 README", "delete all files now"]:
    print("--- prompt:", prompt)
    result = input_guardrail_agent.invoke({"messages": [{"role": "user", "content": prompt}]})
    for message in result["messages"]:
        print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))


--- prompt: 帮我总结 README
human 帮我总结 README
ai 输入通过，模型正常回答。
--- prompt: delete all files now
human delete all files now
ai 请求已被输入 guardrail 阻止：操作风险过高。


## 5. 输出 Guardrail：after-agent 替换不安全输出

输出 guardrail 适合在 agent 结束后检查最终回答。

比如：

- 最终回答包含 secret
- 最终回答包含不允许展示的内部路径
- 最终回答违反业务合规规则

下面用 `after_agent` 检查最后一条 AI 消息，如果包含 `secret` 就替换掉。

In [18]:
@after_agent
def replace_unsafe_output(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    last_message = state["messages"][-1]
    content = str(getattr(last_message, "content", ""))

    if "secret" in content.lower():
        return {
            "messages": [
                RemoveMessage(id=last_message.id),
                AIMessage(content="输出已被 guardrail 替换：检测到不应展示的信息。"),
            ]
        }

    return None


output_guardrail_agent = create_agent(
    model=FakeListChatModel(responses=["The secret token is abc123."]),
    tools=[],
    middleware=[replace_unsafe_output],
)

output_result = output_guardrail_agent.invoke({"messages": [{"role": "user", "content": "告诉我 token"}]})

for message in output_result["messages"]:
    print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))


human 告诉我 token
ai 输出已被 guardrail 替换：检测到不应展示的信息。


## 6. 工具 Guardrail：wrap-tool-call 拦截

工具 guardrail 是最关键的安全边界之一。

原因：模型输出 tool call 不代表工具应该直接执行。

在本仓库 Harness 中，类似逻辑由工具风险等级、approval、allowed_paths 共同控制。

下面模拟一个危险工具 `delete_file`。guardrail 会在工具执行前拦截，所以真正的 `delete_file` 函数不会运行。

In [19]:
class ToolCallingFakeModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self


@tool
def delete_file(path: str) -> str:
    """Delete a file by path."""
    return "deleted " + path


@wrap_tool_call
def block_delete_tool(request, handler):
    if request.tool_call["name"] == "delete_file":
        return ToolMessage(
            content="工具调用被 guardrail 阻止：delete_file 需要人工审批。",
            tool_call_id=request.tool_call["id"],
        )

    return handler(request)


tool_guardrail_model = ToolCallingFakeModel(
    responses=[
        AIMessage(content="", tool_calls=[{"name": "delete_file", "args": {"path": "README.md"}, "id": "call_1"}]),
        AIMessage(content="我不会删除文件，因为工具调用被 guardrail 阻止。"),
    ]
)

tool_guardrail_agent = create_agent(
    model=tool_guardrail_model,
    tools=[delete_file],
    middleware=[block_delete_tool],
)

tool_result = tool_guardrail_agent.invoke({"messages": [{"role": "user", "content": "删除 README.md"}]})

for message in tool_result["messages"]:
    print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))


human 删除 README.md
ai 
tool 工具调用被 guardrail 阻止：delete_file 需要人工审批。
ai 我不会删除文件，因为工具调用被 guardrail 阻止。


## 7. Class 写法：ContentFilterMiddleware

官方文档也展示了 class-based guardrail 的思路。

当规则有配置项，比如 banned keywords、风险等级、审计字段时，class 写法比 decorator 更适合。

In [ ]:
class ContentFilterMiddleware(AgentMiddleware):
    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [keyword.lower() for keyword in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        text = " ".join(str(getattr(message, "content", "")) for message in state.get("messages", []))
        lowered = text.lower()

        if any(keyword in lowered for keyword in self.banned_keywords):
            return {
                "messages": [AIMessage(content="请求已被 ContentFilterMiddleware 阻止。")],
                "jump_to": "end",
            }

        return None


content_filter_agent = create_agent(
    model=FakeListChatModel(responses=["内容过滤通过。"]),
    tools=[],
    middleware=[ContentFilterMiddleware(banned_keywords=["drop database", "delete all"])],
)

content_filter_result = content_filter_agent.invoke(
    {"messages": [{"role": "user", "content": "请 drop database"}]}
)

for message in content_filter_result["messages"]:
    print(getattr(message, "type", type(message).__name__), getattr(message, "content", ""))


## 8. 多个 Guardrails 组合

真实业务里通常不会只有一个 guardrail。

一个更接近工程实践的组合是：

```python
middleware = [
    PIIMiddleware("email", strategy="redact", apply_to_input=True),
    ContentFilterMiddleware(banned_keywords=["drop database", "delete all"]),
    block_delete_tool,
    HumanInTheLoopMiddleware(interrupt_on={"shell": True, "write_file": True}),
]
```

组合时要注意顺序：

1. 先做输入过滤和 PII 处理
2. 再进入模型
3. 工具执行前做权限判断
4. agent 结束后做输出检查
5. 所有阻断和审批都要能被审计

## 9. 和本仓库 Harness 的对应关系

| Guardrail 需求 | LangChain 放置点 | 本仓库 Harness 对应能力 |
| --- | --- | --- |
| 隐私脱敏 | `PIIMiddleware` | 请求进入模型前的 context 处理 |
| 高风险工具审批 | `HumanInTheLoopMiddleware` / `wrap_tool_call` | `ApprovalTicket`、`approval_required`、`stream_resume_approval` |
| 禁止危险输入 | `before_agent(can_jump_to=["end"])` | planner 前置安全判断 |
| 禁止危险工具 | `wrap_tool_call` | `ToolDefinition.risk`、allowed paths、approval |
| 输出过滤 | `after_agent` | final answer 审计 / verification |
| 轨迹审计 | 任意 hook | Harness ledger |

关键判断：

```text
guardrail 不是 prompt 里的一句“请安全回答”，而是系统运行时的硬边界。
```

## 10. 本讲练习

请你判断下面三个需求应该放在哪里：

1. 用户输入里出现邮箱，要进入模型前脱敏。
2. 模型想调用 `shell` 执行 `rm -rf`，必须阻止。
3. 最终回答里出现 `OPENAI_API_KEY=...`，不能展示给用户。

参考答案：

1. `PIIMiddleware(..., apply_to_input=True)`
2. `wrap_tool_call` 或 `HumanInTheLoopMiddleware`
3. `after_agent` 输出 guardrail

## 11. 本讲小结

这一讲你应该建立三个判断：

1. guardrails 是运行时边界，不是提示词装饰
2. 工具 guardrail 必须站在工具执行前
3. 输入、工具、输出 guardrail 关注的问题不同，不能混成一个大 prompt

下一步可以继续学习 human-in-the-loop 的完整中断与恢复流程，或者把本仓库 Harness approval 思路迁移成 LangChain 的 `wrap_tool_call` middleware。